In [ ]:
import requests
from pymongo import MongoClient
import tiktoken
import json

import json

with open("config.json") as f:
    config = json.load(f)

gh_token = config["github_token"]
mongo_connection_string = config["mongo_uri"]

headers = {"Authorization": f"Bearer {gh_token}"}
tokenizer = tiktoken.get_encoding("cl100k_base")

client = MongoClient(mongo_connection_string)
db = client.consensus
collection = db.github_pr_comments

def fetch_pr_comments(owner, repo, pr_number):
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}/comments"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()

def store_comment(comment):
    text = comment["body"]
    tokens = tokenizer.encode(text)

    doc = {
        "source": "github",
        "url": comment["html_url"],
        "repo": f"{owner}/{repo}",
        "type": "pull_request_comment",
        "author": comment["user"]["login"],
        "created_at": comment["created_at"],
        "raw_text": text,
        "tokens": tokens,
        "metadata": {"comment_id": comment["id"]}
    }
    collection.insert_one(doc)

comments = fetch_pr_comments("openrobotics", "reps", 16)
for c in comments:
    store_comment(c)
